# Telco Customer Churn — Case Study 2

This notebook reproduces the five Excel tasks in Python (pandas / matplotlib) so the
whole workflow is reproducible.

**Before running:** place `Telco-Customer-Churn.csv` in the same folder as this notebook.
This is the well-known Kaggle "Telco Customer Churn" dataset, with columns including:

`customerID, gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService,
MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection,
TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod,
MonthlyCharges, TotalCharges, Churn`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

CSV_PATH = "Telco-Customer-Churn.csv"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"'{CSV_PATH}' not found in the working directory. "
        "Place the Telco-Customer-Churn.csv file next to this notebook and re-run."
    )

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

# TotalCharges sometimes contains blank strings in this dataset; coerce to numeric
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["MonthlyCharges"] = pd.to_numeric(df["MonthlyCharges"], errors="coerce")
df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce")

# Standardize Churn to exactly 'Yes'/'No' text
df["Churn"] = df["Churn"].astype(str).str.strip()

print(df.shape)
df.head()


## Task 1 — Count churned vs non-churned customers

Equivalent of filtering `Churn = "Yes"` and `Churn = "No"` and counting rows.


In [ ]:
churn_counts = df["Churn"].value_counts()

churn_yes = int(churn_counts.get("Yes", 0))
churn_no = int(churn_counts.get("No", 0))

print(f"Churn = Yes: {churn_yes:,}")
print(f"Churn = No:  {churn_no:,}")

churn_counts


## Task 2 — Contract type vs Churn (PivotTable equivalent)

Count of churned customers (and total, for context) broken down by `Contract` type.


In [ ]:
contract_churn_pivot = pd.crosstab(df["Contract"], df["Churn"])

# Just the churned-customer counts per contract type, as asked
churned_by_contract = contract_churn_pivot.get("Yes", pd.Series(dtype=int))

print("Churned customers by contract type:")
print(churned_by_contract)

contract_churn_pivot


## Task 3 — Average MonthlyCharges: churned vs non-churned

Equivalent of `=AVERAGEIF(range, criteria, average_range)` for each group.


In [ ]:
avg_monthly_churned = df.loc[df["Churn"] == "Yes", "MonthlyCharges"].mean()
avg_monthly_not_churned = df.loc[df["Churn"] == "No", "MonthlyCharges"].mean()

print(f"Average MonthlyCharges (Churn = Yes): {avg_monthly_churned:,.2f}")
print(f"Average MonthlyCharges (Churn = No):  {avg_monthly_not_churned:,.2f}")


## Task 4 — Bar chart: churned vs non-churned by InternetService

Compares counts of churned vs non-churned customers across each `InternetService`
category (`DSL`, `Fiber optic`, `No`).


In [ ]:
internet_churn = pd.crosstab(df["InternetService"], df["Churn"])
# Make sure both columns exist even if one is missing in a small sample
for col in ["Yes", "No"]:
    if col not in internet_churn.columns:
        internet_churn[col] = 0
internet_churn = internet_churn[["No", "Yes"]]

ax = internet_churn.plot(kind="bar", figsize=(7, 5), color=["#4C72B0", "#C44E52"])
ax.set_title("Churn by Internet Service Type")
ax.set_xlabel("Internet Service")
ax.set_ylabel("Number of Customers")
ax.legend(title="Churn")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("churn_by_internet_service.png", dpi=150)
plt.show()

internet_churn


## Task 5 — Correlation between tenure and Churn

Equivalent of `=CORREL(array1, array2)` after converting `Churn` to a numeric column
(`Yes` → 1, `No` → 0).


In [ ]:
df["Churn_numeric"] = df["Churn"].map({"Yes": 1, "No": 0})

correlation = df["tenure"].corr(df["Churn_numeric"])

print(f"Correlation between tenure and Churn: {correlation:.4f}")

if correlation < -0.3:
    interpretation = (
        "a moderate-to-strong negative correlation: longer-tenured customers are "
        "noticeably less likely to churn."
    )
elif correlation < 0:
    interpretation = (
        "a weak negative correlation: there's a mild tendency for longer-tenured "
        "customers to churn less, but tenure alone doesn't explain much of the variation."
    )
elif correlation == 0:
    interpretation = "no linear relationship between tenure and churn."
else:
    interpretation = (
        "a positive correlation, which would be unusual for this metric — worth "
        "double-checking the Churn encoding."
    )

print("Interpretation:", interpretation)


## Export everything to Excel

Writes `Telco_Churn_Analysis.xlsx` with one sheet per task, plus the bar chart image
embedded, so you also have an Excel-ready copy of every result.


In [ ]:
with pd.ExcelWriter("Telco_Churn_Analysis.xlsx", engine="openpyxl") as writer:
    churn_counts.rename("Count").to_frame().to_excel(writer, sheet_name="Churn_Counts")
    contract_churn_pivot.to_excel(writer, sheet_name="Contract_vs_Churn")
    pd.DataFrame({
        "Group": ["Churned (Yes)", "Not churned (No)"],
        "Average_MonthlyCharges": [avg_monthly_churned, avg_monthly_not_churned],
    }).to_excel(writer, sheet_name="Avg_Monthly_Charges", index=False)
    internet_churn.to_excel(writer, sheet_name="Internet_vs_Churn")
    pd.DataFrame({
        "Metric": ["Correlation (tenure vs Churn)"],
        "Value": [correlation],
        "Interpretation": [interpretation],
    }).to_excel(writer, sheet_name="Tenure_Correlation", index=False)

from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

wb = load_workbook("Telco_Churn_Analysis.xlsx")
ws = wb["Internet_vs_Churn"]
try:
    img = XLImage("churn_by_internet_service.png")
    ws.add_image(img, "F2")
except Exception as e:
    print("Could not embed chart image:", e)

wb.save("Telco_Churn_Analysis.xlsx")
print("Saved Telco_Churn_Analysis.xlsx")
